In [1]:
# Import libraries

import pandas as pd
import numpy as np

print("Libraries loaded")

Libraries loaded


In [2]:
# Load Phase 1 processed data

X_train = pd.read_csv(
    "../../phase1-data-engineering/results/X_train_processed.csv"
)

X_test = pd.read_csv(
    "../../phase1-data-engineering/results/X_test_processed.csv"
)

y_train = pd.read_csv(
    "../../phase1-data-engineering/results/y_train.csv"
).squeeze()

y_test = pd.read_csv(
    "../../phase1-data-engineering/results/y_test.csv"
).squeeze()

print("Data loaded")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Data loaded
X_train: (8000, 12)
X_test: (2000, 12)
y_train: (8000,)
y_test: (2000,)


Check train/test data

In [3]:
print("Training data:")
display(X_train.head())

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())

Training data:


,numerical__amount,numerical__transaction_hour,numerical__foreign_transaction,numerical__location_mismatch,numerical__device_trust_score,numerical__velocity_last_24h,numerical__cardholder_age,categorical__merchant_category_Clothing,categorical__merchant_category_Electronics,categorical__merchant_category_Food,categorical__merchant_category_Grocery,categorical__merchant_category_Travel
0,-0.769255,-0.808001,-0.330549,3.280961,0.930866,-0.005472,0.507084,0.0,0.0,1.0,0.0,0.0
1,-0.677049,-0.808001,-0.330549,-0.304789,-0.930308,-0.005472,-1.696975,0.0,1.0,0.0,0.0,0.0
2,-0.750482,-1.675479,-0.330549,-0.304789,-1.674777,-0.005472,-1.229448,0.0,0.0,0.0,1.0,0.0
3,1.267417,1.649853,-0.330549,-0.304789,1.628806,-0.700342,-0.628341,1.0,0.0,0.0,0.0,0.0
4,-0.888477,-1.675479,-0.330549,-0.304789,1.349630,-0.700342,-1.029079,0.0,0.0,1.0,0.0,0.0



Training shape: (8000, 12)
Testing shape: (2000, 12)

Training target:
is_fraud
0    7879
1     121
Name: count, dtype: int64

Testing target:
is_fraud
0    1970
1      30
Name: count, dtype: int64


In [4]:
#check missing
print("Missing values in training data:")
print(X_train.isna().sum().sum())

print("\nMissing values in testing data:")
print(X_test.isna().sum().sum())

Missing values in training data:
0

Missing values in testing data:
0


In [5]:
# set up cross validation and metrics

from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

print("5-fold stratified CV ready")

5-fold stratified CV ready


In [6]:
# logistic regression

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_cv = cross_validate(
    lr_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": lr_cv["test_accuracy"].mean(),
    "Precision": lr_cv["test_precision"].mean(),
    "Recall": lr_cv["test_recall"].mean(),
    "F1": lr_cv["test_f1"].mean(),
    "ROC_AUC": lr_cv["test_roc_auc"].mean(),
    "PR_AUC": lr_cv["test_pr_auc"].mean()
}

lr_results

{'Model': 'Logistic Regression',
 'Accuracy': np.float64(0.990375),
 'Precision': np.float64(0.7709316770186335),
 'Recall': np.float64(0.5103333333333333),
 'F1': np.float64(0.6046346068791888),
 'ROC_AUC': np.float64(0.9907437607767303),
 'PR_AUC': np.float64(0.6941708567168656)}

In [7]:
# random forest

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_cv = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

rf_results = {
    "Model": "Random Forest",
    "Accuracy": rf_cv["test_accuracy"].mean(),
    "Precision": rf_cv["test_precision"].mean(),
    "Recall": rf_cv["test_recall"].mean(),
    "F1": rf_cv["test_f1"].mean(),
    "ROC_AUC": rf_cv["test_roc_auc"].mean(),
    "PR_AUC": rf_cv["test_pr_auc"].mean()
}

rf_results

{'Model': 'Random Forest',
 'Accuracy': np.float64(0.9949999999999999),
 'Precision': np.float64(1.0),
 'Recall': np.float64(0.6696666666666666),
 'F1': np.float64(0.8005340547144291),
 'ROC_AUC': np.float64(0.9995717005076143),
 'PR_AUC': np.float64(0.9812773501492327)}

In [10]:
%pip install xgboost

  Using cached xgboost-3.4.1-py3-none-macosx_12_0_arm64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-macosx_12_0_arm64.whl (2.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [12]:
# xgboost

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_cv = cross_validate(
    xgb_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

xgb_results = {
    "Model": "XGBoost",
    "Accuracy": xgb_cv["test_accuracy"].mean(),
    "Precision": xgb_cv["test_precision"].mean(),
    "Recall": xgb_cv["test_recall"].mean(),
    "F1": xgb_cv["test_f1"].mean(),
    "ROC_AUC": xgb_cv["test_roc_auc"].mean(),
    "PR_AUC": xgb_cv["test_pr_auc"].mean()
}

xgb_results

{'Model': 'XGBoost',
 'Accuracy': np.float64(0.999),
 'Precision': np.float64(0.992),
 'Recall': np.float64(0.942),
 'F1': np.float64(0.9660980573543014),
 'ROC_AUC': np.float64(0.9999422524373539),
 'PR_AUC': np.float64(0.9970995804195804)}

In [14]:
%pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 1.2 MB/s  0:00:01 eta 0:00:010m
Note: you may need to restart the kernel to use updated packages.


In [16]:
# lightgbm

from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

lgbm_cv = cross_validate(
    lgbm_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

lgbm_results = {
    "Model": "LightGBM",
    "Accuracy": lgbm_cv["test_accuracy"].mean(),
    "Precision": lgbm_cv["test_precision"].mean(),
    "Recall": lgbm_cv["test_recall"].mean(),
    "F1": lgbm_cv["test_f1"].mean(),
    "ROC_AUC": lgbm_cv["test_roc_auc"].mean(),
    "PR_AUC": lgbm_cv["test_pr_auc"].mean()
}

lgbm_results

{'Model': 'LightGBM',
 'Accuracy': np.float64(0.999125),
 'Precision': np.float64(1.0),
 'Recall': np.float64(0.9416666666666668),
 'F1': np.float64(0.9690522243713733),
 'ROC_AUC': np.float64(0.9998096446700508),
 'PR_AUC': np.float64(0.9936965488215488)}

In [18]:
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 1.7 MB/s  0:00:17m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]1/2 [catboost]
Note: you may need to restart the kernel to use updated packages.


In [19]:
# catboost

from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=False
)

cat_cv = cross_validate(
    cat_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

cat_results = {
    "Model": "CatBoost",
    "Accuracy": cat_cv["test_accuracy"].mean(),
    "Precision": cat_cv["test_precision"].mean(),
    "Recall": cat_cv["test_recall"].mean(),
    "F1": cat_cv["test_f1"].mean(),
    "ROC_AUC": cat_cv["test_roc_auc"].mean(),
    "PR_AUC": cat_cv["test_pr_auc"].mean()
}

cat_results

{'Model': 'CatBoost',
 'Accuracy': np.float64(0.999625),
 'Precision': np.float64(0.9923076923076923),
 'Recall': np.float64(0.9833333333333334),
 'F1': np.float64(0.9875677930746767),
 'ROC_AUC': np.float64(0.9996615905245345),
 'PR_AUC': np.float64(0.993632183908046)}

In [25]:
# put the model results together

results_df = pd.DataFrame([
    lr_results,
    rf_results,
    xgb_results,
    lgbm_results,
    cat_results
])

results_df = results_df.sort_values(
    "PR_AUC",
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,XGBoost,0.9990,0.9920,0.9420,0.9661,0.9999,0.9971
1,LightGBM,0.9991,1.0000,0.9417,0.9691,0.9998,0.9937
2,CatBoost,0.9996,0.9923,0.9833,0.9876,0.9997,0.9936
3,Random Forest,0.9950,1.0000,0.6697,0.8005,0.9996,0.9813
4,Logistic Regression,0.9904,0.7709,0.5103,0.6046,0.9907,0.6942


In [26]:
# pick the strongest baseline

best_model = results_df.iloc[0]

print("Best baseline model:", best_model["Model"])
print("PR-AUC:", round(best_model["PR_AUC"], 4))
print("ROC-AUC:", round(best_model["ROC_AUC"], 4))
print("F1:", round(best_model["F1"], 4))
print("Precision:", round(best_model["Precision"], 4))
print("Recall:", round(best_model["Recall"], 4))

Best baseline model: XGBoost
PR-AUC: 0.9971
ROC-AUC: 0.9999
F1: 0.9661
Precision: 0.992
Recall: 0.942


In [27]:
# save the comparison

import os

os.makedirs("../results", exist_ok=True)

results_df.to_csv(
    "../results/day5_model_comparison.csv",
    index=False
)

print("Model comparison saved")

Model comparison saved
